# Unsupervised Fraud Detection via Behavioural Anomaly Analysis

This notebook applies **unsupervised anomaly detection** to identify potentially fraudulent behaviour in a highly imbalanced transaction dataset.

The analysis is conducted in stages with increasing behavioural context:
- **Transaction-level analysis**, scoring individual transactions  
- **Account-level behavioural analysis**, aggregating transactions across each account's history  
- **Account x day behavioural analysis**, examining daily behaviour to capture short-term irregular activity  

At each stage, models are trained **only on non-fraud data** to learn patterns of normal behaviour. Transactions or behavioural profiles that deviate from these patterns receive higher anomaly scores and are treated as higher risk.

Evaluation focuses on **fraud concentration at top-K cut-offs**, assessing how much fraud is captured among the highest-ranked alerts relative to the baseline fraud rate.

The objective is to gain behavioural insight and support **risk prioritisation through alert ranking**, rather than binary fraud classification.


## 1.Setup Environment and Load Data

In [ ]:
#mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
#import required libraries

import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt
import joblib

from datasets import load_dataset
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OrdinalEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.ensemble import IsolationForest

In [ ]:
#load saved sampled dataset

data_folder='/content/drive/MyDrive/hybrid-fraud-risk-prioritisation/data'
df=pd.read_parquet(f"{data_folder}/sample_5M_df.parquet")

In [ ]:
df.info()

In [ ]:
df.shape

## 2.Unsupervised Learning for Transaction-Level Anomaly Detection

### 2.1.Define Features and Sample Non-Fraud Training Data

In [ ]:
#exclude label and flag columns, and remove ratio-based features to prevent scale-dominated anomaly scores
exclude_from_X=[
    'isFraud',
    'isFlaggedFraud',
    'amount_to_oldbalance_ratio',
    'amount_to_destbalance_ratio'
]

#define the final feature set used for unsupervised anomaly detection
feature_cols = [
    'step',
    'type',
    'amount',
    'oldbalanceOrg',
    'newbalanceOrig',
    'oldbalanceDest',
    'newbalanceDest',
    'day',
    'hour',
    'balance_change_orig',
    'balance_change_dest',
    'type_high_amount',
    'part_of_day',
    'amount_quantile'
]

#ensure excluded features are not included in the model input
feature_cols=[c for c in feature_cols if c not in exclude_from_X]

#sample a subset of non-fraud transactions to train the model on normal behaviour
nonfraud_df=df[df['isFraud']==0]
nonfraud_train=nonfraud_df.sample(
    n=min(300000, len(nonfraud_df)),
)
print('Non fraud train size:', nonfraud_train.shape)

### 2.2.Train Isolation Forest on Transaction-Level Non-Fraud Behaviour

In [ ]:
#separate categorical and numerical features for preprocessing
cat_cols=['type', 'part_of_day']
numeric_cols=[c for c in feature_cols if c not in cat_cols]

#apply ordinal encoding to categorical features and standardise numerical features
preprocess=ColumnTransformer(
    transformers=[
        ('cat', OrdinalEncoder(
            handle_unknown='use_encoded_value',
            unknown_value=-1
        ), cat_cols),
        ('num', StandardScaler(), numeric_cols)
    ], remainder='drop'
)

#build an Isolation Forest pipeline for unsupervised anomaly detection
model=Pipeline(steps=[
    ('prep', preprocess),
    ('iso', IsolationForest(
        n_estimators=300,
        max_samples=256,
        contamination=0.001,
        random_state=42,
        n_jobs=-1
    ))
])

#train the model using non-fraud transactions to learn normal behaviour patterns
model.fit(nonfraud_train[feature_cols])

### 2.3.Analyse Fraud Concentration in Ranked Accounts

In [ ]:
#computer anomaly scores for all trasactions
df['anomaly_score'] = -model.decision_function(df[feature_cols])

#rank transactions from most anomalous to least anomalous
df_ranked = df.sort_values('anomaly_score', ascending=False)

In [ ]:
#measure how much fraud is concentrated in the top-ranked alerts compared to baseline
n=len(df)

baseline=df['isFraud'].mean()
print(f'Baseline fraud rate: {baseline:.4f}\n')

ks=[100, 500, int(0.001*n), int(0.005*n), int(0.01*n)]

for k in ks:
  fr=df_ranked.head(k)['isFraud'].mean()
  enrichment=fr/baseline
  print(f'Top {k:>5} | Fraud_rate: {fr:.4f} | Enrichment: {enrichment:.1f}x')

## 3.Unsupervised Learning for Account-Level Behavioural Anomaly Detection

### 3.1.Aggregate Transactions into Account-Level Behaviour Profiles

In [ ]:
#aggregate transaction-level into account-level behavioural features

behaviour_df=(
    df
    .groupby('nameOrig')
    .agg(
        trans_count=('amount', 'count'),
        total_amount=('amount', 'sum'),
        avg_amount=('amount', 'mean'),
        max_amount=('amount', 'max'),
        avg_balance_orig=('oldbalanceOrg', 'mean'),
        avg_balance_change=('balance_change_orig', 'mean'),
        unique_destinations=('nameDest', 'nunique'),
        active_hours=('hour', 'nunique'),
        fraud_count=('isFraud', 'sum')
    )
    .reset_index()
)

In [ ]:
#mark an account as fraudulent if at least one fraud transaction is present in its history

behaviour_df['isFraud']=(behaviour_df['fraud_count']>0).astype(int)

In [ ]:
behaviour_df.head()

In [ ]:
#define behavioural features to model normal account activity

behaviour_features=[
    'trans_count',
    'total_amount',
    'avg_amount',
    'max_amount',
    'avg_balance_orig',
    'avg_balance_change',
    'unique_destinations',
    'active_hours',
    'fraud_count'
]

### 3.2.Train Isolation Forest on Non-Fraud Account-Level Behaviour

In [ ]:
#define an Isolation Forest pipeline to model non-fraud account-level behaviour

model=Pipeline(steps=[
    ('scale', StandardScaler()),
    ('iso', IsolationForest(
        n_estimators=300,
        max_samples=256,
        contamination=0.001,
        random_state=42,
        n_jobs=-1
    ))
])

In [ ]:
#fit the model using only non-fraud accounts to learn normal behaviour patterns

normal_accounts=behaviour_df[behaviour_df['isFraud']==0]
model.fit(normal_accounts[behaviour_features])

### 3.3.Analyse Fraud Concentration in Ranked Accounts

In [ ]:
#score and rank account-days by anomaly level (higher score = higher risk)

behaviour_df['anomaly_score']=-model.decision_function(behaviour_df[behaviour_features])
behaviour_ranked=behaviour_df.sort_values('anomaly_score', ascending=False)

In [ ]:
#evaluate how much fraud is concentrated at different top-K cut-offs compared to the overall baseline rate

n=len(behaviour_df)

ks=[
    100,
    500,
    1000,
    int(0.001*n),
    int(0.005*n),
    int(0.01*n)
]

baseline=behaviour_df['isFraud'].mean()
print(f'Baseline fraud rate: {baseline:.4f}\n')

for k in ks:
  fraud_rate=behaviour_ranked.head(k)['isFraud'].mean()
  enrichment=fraud_rate/baseline

  print(
      f'Top {k: >5} | '
      f'Fraud rate: {fraud_rate:.4f} | '
      f'Enrichment: {enrichment:.1f}x'
  )

## 4.Unsupervised Learning for Account x Day Behavioural Anomaly Detection

### 4.1.Aggregate Transactions into Account x Day Behaviour Profiles

In [ ]:
#aggregate transactions into daily behaviour profiles for each account

behaviour_day_df=(
    df
    .groupby(['nameOrig', 'day'])
    .agg(
        trans_count_day=('amount', 'count'),
        total_amount_day=('amount', 'sum'),
        avg_amount_day=('amount', 'mean'),
        max_amount_day=('amount', 'max'),
        unique_destinations_day=('nameDest', 'nunique'),
        active_hours_day=('day', 'nunique'),
        fraud_count_day=('isFraud', 'sum')
    )
    .reset_index()
)

In [ ]:
#mark an account_day as fraudulent if any fraud occurred on that day

behaviour_day_df['isFraud']=(behaviour_day_df['fraud_count_day']>0).astype(int)

In [ ]:
behaviour_day_df.head()

In [ ]:
#define behavioural features used to model normal account x day activity

behaviour_day_features=[
    'trans_count_day',
    'total_amount_day',
    'avg_amount_day',
    'max_amount_day',
    'unique_destinations_day',
    'active_hours_day'
]

#select only non-fraud account x days for training
normal_days=behaviour_day_df[behaviour_day_df['isFraud']==0]

### 4.2.Train Isolation Forest on Non-Fraud Account x Day Behaviour

In [ ]:
#define an Isolation Forest pipeline to learn normal account-day behaviour
model=Pipeline(steps=[
    ('scale', StandardScaler()),
    ('iso', IsolationForest(
        n_estimators=300,
        max_samples=256,
        contamination=0.001,
        random_state=42,
        n_jobs=-1
    ))
])

#fit the model using normal account-day behaviour only
model.fit(normal_days[behaviour_day_features])

### 4.3.Analyse Fraud Concentration in Ranked Account x Days

In [ ]:
#score and rank account x days by anomaly level (higher score = higher risk)

behaviour_day_df['anomaly_score']=-model.decision_function(behaviour_day_df[behaviour_day_features])
behaviour_day_ranked=behaviour_day_df.sort_values('anomaly_score', ascending=False)

In [ ]:
#evaluate how much fraud is concentrated at different top-K cut-offs compared to the overall baseline rate

n=len(behaviour_day_df)

ks=[
    100,
    500,
    1000,
    int(0.001*n),
    int(0.005*n),
    int(0.01*n)
]

baseline=behaviour_day_df['isFraud'].mean()
print(f'Baseline fraud rate: {baseline:.4f}\n')

for k in ks:
  fraud_rate=behaviour_day_ranked.head(k)['isFraud'].mean()
  enrichment=fraud_rate/baseline

  print(
      f'Top {k: >5} | '
      f'Fraud rate: {fraud_rate:.4f} | '
      f'Enrichment: {enrichment:.1f}x'
  )

**Notes:**

Three unsupervised representations were evaluated: **transaction-level**, **account-level**, and **account x day** behaviour.

At the transaction level, anomaly ranking showed limited effectiveness. Fraud did not appear among the top-ranked transactions, and enrichment remained close to the baseline across most cut-offs, indicating that individual transactions lack sufficient behavioural context for strong prioritisation.

At the account level, aggregation over an account's full history improved fraud detection at larger review scopes, with enrichment rising to approximately **2.1x-2.4x**. However, fraud was still absent from the very top of the ranking, suggesting that long-term aggregation dilutes short-term fraud signals and delays effective prioritisation.

In contrast, the **account x day** representation produced the strongest and most actionable results. Fraud was highly concentrated among the top-ranked account-days, achieving an enrichment of **3.5x** at the top 100 and remaining consistently above baseline at larger cut-offs. This indicates that fraudulent activity in the dataset tends to occur within short time windows rather than as persistently abnormal behaviour.

Overall, these results show that **time-windowed behavioural representations provide the most effective basis for unsupervised fraud prioritisation**, outperforming both transaction-level analysis and long-term account aggregation.


## 5.Export Account x Day Anomaly Scores for Downstream Use

In [ ]:
#examine the structure of the account-day anomaly results

behaviour_day_df.info()

In [ ]:
#extract key identifiers and anomaly scores for downstream hybrid fraud prioritisation

anomaly_results=behaviour_day_df[
    ['nameOrig','day', 'anomaly_score']
].copy()

In [ ]:
anomaly_results.shape

In [ ]:
#check for missing values in the extract anomaly results

anomaly_results.isnull().sum()

In [ ]:
#save the anomaly scores for later use in hybrid fraud prioritisation
output_dir='/content/drive/MyDrive/hybrid-fraud-risk-prioritisation/outputs'
os.makedirs(output_dir, exist_ok=True)

anomaly_results.to_parquet(
    f'{output_dir}/anomaly_results.parquet', index=False
)